# M07 — Parametric Normal and PCA Monte Carlo

This notebook reproduces the M07 comparison from committed reusable code. It downloads official GSW data, selects the latest complete curve date, compares three risk models, and inspects PCA residual risk. All Gaussian benchmarks are physical-measure (`P`) risk models rather than risk-neutral pricing calibrations.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_ROOT = Path('/content/market_risk') if 'google.colab' in sys.modules else Path.cwd().resolve()
if 'google.colab' in sys.modules:
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Run the accepted pipeline

The command uses 500, 750, and 1,250-day windows and 100,000 fixed-seed paths per window. Generated data stays in the temporary runtime.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_m07_model_comparison.py'], check=True)

In [ ]:
import json
import pandas as pd

report_path = max((PROJECT_ROOT / 'data' / 'audit').glob('m07_model_comparison_*.json'))
report = json.loads(report_path.read_text())
assert report['status'] == 'PASS', report['checks']
display(pd.DataFrame(report['primary_750_day_results']).pivot(index='method', columns=['measure', 'confidence_level'], values='value').round(2))
print('Valuation date:', report['portfolio']['valuation_date'])
print('All checks passed:', all(report['checks'].values()))

In [ ]:
curve_variance = pd.DataFrame(report['primary_750_day_curve_variance'])
portfolio_variance = pd.DataFrame(report['primary_750_day_portfolio_variance'])
attribution = curve_variance.merge(portfolio_variance, on='factor')
display(attribution.style.format({'explained_variance_ratio': '{:.2%}', 'portfolio_variance_share': '{:.2%}'}))
ax = attribution.set_index('factor')[['explained_variance_ratio', 'portfolio_variance_share']].plot.bar(figsize=(9, 4), title='750-day PCA variance attribution')
ax.set_ylabel('Share')
ax.set_xlabel('Factor group')

## Interpretation boundary

The first three factors are shown separately for interpretation, while all remaining components are retained in every Monte Carlo path. Similar Parametric Normal and PCA Monte Carlo results are expected because both use the same Gaussian covariance. Historical Simulation can differ materially in the tail because it preserves the observed empirical distribution.